# 🚀 Notebook 02: VLM QLoRA Fine-Tuning on Google Colab A100 / T4
### Melatih Qwen2-VL-2B-Instruct untuk Visual Hardware & Automotive Diagnostics

Notebook ini menjalankan pelatihan **QLoRA 4-bit** yang sangat efisien. Model base dimuat dalam kuantisasi NormalFloat4 (NF4) sehingga hanya membutuhkan ~8-12 GB VRAM dan dapat menyelesaikan training dalam hitungan jam di GPU A100 Colab.

In [ ]:
# 1. Hubungkan Google Drive (Wajib untuk auto-save model)
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/Edge-VLM-Project'
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints/qwen2_vl_lora"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Direktori Checkpoint: {CHECKPOINT_DIR}")

In [ ]:
# 2. Verifikasi Spesifikasi GPU
!nvidia-smi

In [ ]:
# 3. Pasang library Hugging Face & QLoRA terbaru
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q peft bitsandbytes accelerate datasets trl qwen-vl-utils

In [ ]:
# 4. Inisialisasi Model Qwen2-VL-2B-Instruct dengan 4-bit NF4
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("⏳ Memuat processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=1280*28*28)

print("⏳ Memuat base model dengan 4-bit Quantization...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("✅ LoRA Adapter siap dilatih!")

In [ ]:
# 5. Konfigurasi Training & Simpan Adapter ke Google Drive
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    report_to="none"
)

print("✅ Training configuration successfully registered!")


In [ ]:
# 6. Simpan LoRA Adapter Final ke Google Drive
FINAL_ADAPTER_DIR = f"{WORK_DIR}/models/edge_vlm_final_adapter"
model.save_pretrained(FINAL_ADAPTER_DIR)
processor.save_pretrained(FINAL_ADAPTER_DIR)
print(f"🎉 Adapter LoRA berhasil disimpan permanen di: {FINAL_ADAPTER_DIR}")